In [1]:
import numpy as np
import xarray as xr

from openghg_inversions.basis.basis_functions import BasisFunctions
from openghg_inversions.basis.layout import BasisLayout, BasisPartition


coords = {"lat": [51.0, 52.0], "lon": [-2.0, -1.0]}
inner_labels = xr.DataArray(
    [[1, 2], [0, 0]],
    dims=("lat", "lon"),
    coords=coords,
    name="inner_labels",
)
remainder_labels = xr.DataArray(
    [[0, 0], [1, 1]],
    dims=("lat", "lon"),
    coords=coords,
    name="remainder_labels",
)

inner = BasisPartition(name="generated_inner", labels=inner_labels, group="inner")
remainder = BasisPartition(
    name="explicit_remainder",
    labels=remainder_labels,
    group="outer",
)
layout = BasisLayout(partitions=(inner, remainder), state_dim="region")
result = layout.to_flat_basis()
result.basis_flat.values

array([[1, 2],
       [3, 3]])

In [2]:
result.state_metadata[["basis_group", "basis_partition", "region_in_partition"]]

<xarray.Dataset> Size: 96B
Dimensions:              (basis_label: 3)
Coordinates:
  * basis_label          (basis_label) int64 24B 1 2 3
Data variables:
    basis_group          (basis_label) object 24B 'inner' 'inner' 'outer'
    basis_partition      (basis_label) object 24B 'generated_inner' ... 'expl...
    region_in_partition  (basis_label) int64 24B 1 2 1
Attributes:
    state_dim:  region

In [3]:
try:
    BasisLayout(partitions=(inner,), state_dim="region").to_flat_basis()
except ValueError as error:
    uncovered_error = str(error)
    print(uncovered_error)

BasisLayout partitions leave 2 grid cells unmapped.


In [4]:
flux = xr.DataArray(
    np.ones((2, 2, 1)),
    dims=("lat", "lon", "time"),
    coords={**coords, "time": ["2020-01-01"]},
    name="flux",
)
basis_functions = BasisFunctions.from_flat_basis(
    result.basis_flat,
    flux,
    region_labels="range0",
    operator_kwargs={
        "state_dim": "region",
        "state_metadata": result.state_metadata,
    },
)
matrix = basis_functions.operator.basis_matrix
print("Raw basis labels:", result.state_metadata.basis_label.values)
print("Operator state labels:", matrix.region.values)
matrix.coords.to_dataset()[
    ["basis_group", "basis_partition", "region_in_partition"]
].compute()

Raw basis labels: [1 2 3]
Operator state labels: [0 1 2]


<xarray.Dataset> Size: 96B
Dimensions:              (region: 3)
Coordinates:
  * region               (region) int64 24B 0 1 2
    basis_group          (region) object 24B 'inner' 'inner' 'outer'
    basis_partition      (region) object 24B 'generated_inner' ... 'explicit_...
    region_in_partition  (region) int64 24B 1 2 1
Data variables:
    *empty*

In [5]:
state = xr.DataArray(
    [0.9, 1.1, 1.0],
    dims="region",
    coords={
        "region": matrix.region.values,
        "basis_group": ("region", matrix.basis_group.values),
        "basis_partition": ("region", matrix.basis_partition.values),
        "region_in_partition": ("region", matrix.region_in_partition.values),
    },
    name="scaling",
)
inner_state = state.where(state.basis_group == "inner", drop=True)
outer_state = state.where(state.basis_group == "outer", drop=True)
print("Inner states:", inner_state.region.values)
print("Outer states:", outer_state.region.values)
print("Outer partition:", outer_state.basis_partition.item())

Inner states: [0 1]
Outer states: [2]
Outer partition: explicit_remainder


In [6]:
restored = BasisFunctions.from_datatree(basis_functions.to_datatree())
restored_matrix = restored.operator.basis_matrix
{
    "region": restored_matrix.region.equals(matrix.region),
    "basis_group": restored_matrix.basis_group.equals(matrix.basis_group),
    "region_in_partition": restored_matrix.region_in_partition.equals(
        matrix.region_in_partition
    ),
}

{'region': True, 'basis_group': True, 'region_in_partition': True}